# 01 · Study population: participants with a smoking history

Participants are included if they meet either EHR criterion at any encounter:
1. a TUD / nicotine-dependence diagnosis (ICD-9-CM 305.1, ICD-10-CM F17.2), or
2. a current/former smoking status recorded at a clinical encounter (SNOMED CT / LOINC observations).

Each person's earliest qualifying date is kept and the study window is applied.

## 1. EHR diagnoses: tobacco use disorder / nicotine dependence

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "Test1" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_30337041_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (1568179, 1568180, 44819867, 44820247, 44831104, 44831391, 44832537, 44837247, 45533085, 45534323, 45538046, 45541762, 45542821, 45551415, 45552453, 45552454, 45560971, 45562025, 45565769, 45566810, 45568007, 45571741, 45571742, 45571743, 45575487, 45575488, 45576521, 45577675, 45580343, 45581435, 45586219, 45589999, 45591115, 45591116, 45594845, 45595884, 45595885, 45600719, 45600720, 45604482, 45605476, 45606676, 45609261)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_30337041_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_30337041",
  "condition_30337041_*.csv")
message(str_glue('The data will be written to {condition_30337041_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_30337041_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_30337041_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_30337041_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
EHR_smokers <- read_bq_export_from_workspace_bucket(condition_30337041_path)

In [ ]:
# Concept list for Supplementary Table S1 (concept codes only, no counts)
EHR_source_concepts <- EHR_smokers %>%
  distinct(source_concept_name, source_concept_code, source_vocabulary) %>%
  arrange(source_vocabulary, source_concept_code)
write_csv(EHR_source_concepts, "supp_table_S1_condition_concepts.csv")

In [ ]:
EHR_smokers <- EHR_smokers %>%
  group_by(person_id) %>%
  arrange(condition_start_datetime) %>%
  slice(1) %>%
  ungroup() %>%
  select(person_id, condition_start_datetime, 
         standard_concept_name, standard_concept_code,source_concept_code, source_concept_name) %>%
  collect()

n_distinct(EHR_smokers$person_id)

## 2. EHR observations: recorded smoking status

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "Test1" for domain "observation" and was generated for All of Us Controlled Tier Dataset v8
dataset_05045737_observation_sql <- paste("
    SELECT
        observation.person_id,
        observation.observation_concept_id,
        o_standard_concept.concept_name as standard_concept_name,
        o_standard_concept.concept_code as standard_concept_code,
        o_standard_concept.vocabulary_id as standard_vocabulary,
        observation.observation_datetime,
        observation.observation_type_concept_id,
        o_type.concept_name as observation_type_concept_name,
        observation.value_as_number,
        observation.value_as_string,
        observation.value_as_concept_id,
        o_value.concept_name as value_as_concept_name,
        observation.qualifier_concept_id,
        o_qualifier.concept_name as qualifier_concept_name,
        observation.unit_concept_id,
        o_unit.concept_name as unit_concept_name,
        observation.visit_occurrence_id,
        o_visit.concept_name as visit_occurrence_concept_name,
        observation.observation_source_value,
        observation.observation_source_concept_id,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary,
        observation.unit_source_value,
        observation.qualifier_source_value,
        observation.value_source_concept_id,
        observation.value_source_value,
        observation.questionnaire_response_id 
    FROM
        ( SELECT
            * 
        FROM
            `observation` observation 
        WHERE
            (
                observation_concept_id IN (4010813, 40766945, 4132133, 4274663, 4282779, 43054909, 903657, 903659, 903661, 903662, 903664)
            )) observation 
    LEFT JOIN
        `concept` o_standard_concept 
            ON observation.observation_concept_id = o_standard_concept.concept_id 
    LEFT JOIN
        `concept` o_type 
            ON observation.observation_type_concept_id = o_type.concept_id 
    LEFT JOIN
        `concept` o_value 
            ON observation.value_as_concept_id = o_value.concept_id 
    LEFT JOIN
        `concept` o_qualifier 
            ON observation.qualifier_concept_id = o_qualifier.concept_id 
    LEFT JOIN
        `concept` o_unit 
            ON observation.unit_concept_id = o_unit.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON observation.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` o_visit 
            ON v.visit_concept_id = o_visit.concept_id 
    LEFT JOIN
        `concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
observation_05045737_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "observation_05045737",
  "observation_05045737_*.csv")
message(str_glue('The data will be written to {observation_05045737_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_05045737_observation_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  observation_05045737_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {observation_05045737_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), observation_type_concept_name = col_character(), value_as_string = col_character(), value_as_concept_name = col_character(), qualifier_concept_name = col_character(), unit_concept_name = col_character(), visit_occurrence_concept_name = col_character(), observation_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), unit_source_value = col_character(), qualifier_source_value = col_character(), value_source_value = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
OB_smokers <- read_bq_export_from_workspace_bucket(observation_05045737_path)

Keep records whose value (or, when the value is blank, whose concept) explicitly asserts current or former smoking.

In [ ]:
# 1. Whitelist of VALUES (Explicit assertions)
target_smoking_values <- c(
  # --- Current / Active ---
  "Cigarette smoker",
  "Cigar smoker",
  "Pipe smoker",
  "Smoker",
  "Current smoker",
  "Current every day smoker",
  "Current some day smoker",
  "Every day",
  "Smokes tobacco daily",
  "Heavy tobacco smoker",
  "Light tobacco smoker",
  "Occasional tobacco smoker",
  "Chain smoker",
  "Chews tobacco",
  "Snuff tobacco",
  "Light cigarette smoker (1-9 cigs/day)",
  "Light cigarette smoker (1-9 cigarettes per day)",
  "Heavy cigarette smoker (20-39 cigarettes per day)",
  "Tobacco or its derivatives user",
  "Yes", # ONLY if the concept name is "Cigarette Smoker" (Handled in filter logic below)
  "Present",
  
  # --- Nicotine Dependence Codes (Strong evidence of history/current) ---
  "Nicotine dependence, cigarettes, uncomplicated",
  "Nicotine dependence, cigarettes, with other nicotine-induced disorders",
  "Nicotine dependence, cigarettes, with unspecified nicotine-induced disorders",
  "Nicotine dependence, cigarettes, with withdrawal",
  
  # --- History / Former (Since you are building a longitudinal cohort) ---
  "Ex-smoker",
  "Former smoker",
  "Nicotine dependence, cigarettes, in remission",
  "History of event"
)

# 2. Whitelist of CONCEPTS (Implied assertions when value is missing)
# Only add concepts here where the name itself = "Smoker"
strong_smoking_concepts <- c(
  "OMOP5181838", # Cigarette smoker
  "OMOP5181846", # Cigar smoker
  "OMOP5181840", # Light cigarette smoker
  "OMOP5181842", # Heavy cigarette smoker
  "OMOP5181843", # Very heavy cigarette smoker
  "64197008",     # Smoke
  "64234-8",     #Current smoker
  "102407002"    #Tobacco smoker
)

# 3. Filter with OR logic
clean_smokers <- OB_smokers %>%
  filter(
    # Condition A: The VALUE is explicitly in our whitelist
    (value_as_concept_name %in% target_smoking_values) |
      
    # Condition B: The VALUE is Blank/Null, BUT the CONCEPT is strong
    (
      (is.na(value_as_concept_name) | value_as_concept_name == "") &
      standard_concept_code %in% strong_smoking_concepts
    )
  ) %>%
  # 4. Cleanup: Remove "Yes" if it was attached to a generic "Status" concept
  filter(
    !(value_as_concept_name %in% c("Yes", "Present") & 
      standard_concept_name == "Tobacco smoking status")
  )

n_distinct(clean_smokers$person_id)

# Check the result
table(clean_smokers$value_as_concept_name)

In [ ]:
smokers_earliest <- clean_smokers %>%
  # Group by person
  group_by(person_id) %>%
  # Get the earliest observation date for each person
  arrange(observation_datetime) %>%
  slice(1) %>%
  ungroup() %>%
  # Select relevant columns
  select(person_id, observation_datetime, 
         standard_concept_name, standard_concept_code,
         value_as_concept_name) %>%
  collect()

## 3. Combine both sources and apply the study window

In [ ]:
library(dplyr)
library(lubridate)
smokers_from_OB <- smokers_earliest %>%
  rename(earliest_date = observation_datetime) %>%
  mutate(source = "Observation")

smokers_from_EHR <- EHR_smokers %>%
  rename(earliest_date = condition_start_datetime) %>%
  mutate(source = "Condition")

# Combine both datasets
combined_smokers <- bind_rows(smokers_from_OB, smokers_from_EHR)

In [ ]:
final_smokers <- combined_smokers %>%
  group_by(person_id) %>%
  arrange(earliest_date) %>%
  slice(1) %>%
  ungroup()